In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import torch
import yaml
from torch.utils.data import DataLoader

In [2]:
def find_project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "configs" / "config.yaml").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Could not find configs/config.yaml")


ROOT = find_project_root()
sys.path.insert(0, str(ROOT.parent))

from ebm_unlearning.src.data.dataset import DatasetSpec, load_dataset
from ebm_unlearning.src.models.ebm import EnergyModel
from ebm_unlearning.src.training.pretrain import PretrainConfig, pretrain_ebm
from ebm_unlearning.src.utils.logging import setup_logger
from ebm_unlearning.src.utils.seed import set_seed
from ebm_unlearning.src.utils.tracking import make_tracker

with open(ROOT / "configs" / "config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

set_seed(int(cfg["seed"]))

device = torch.device(cfg.get("device", "cpu"))

from datetime import datetime

logger = setup_logger("pretrain", log_file=str(ROOT / "outputs" / "logs" / "pretrain.log"))
run_id = datetime.now().strftime("%Y%m%d-%H%M%S")
tracker = make_tracker(
    "tensorboard",
    log_dir=str(ROOT / "outputs" / "tensorboard" / cfg["data"]["dataset"] / "pretrain" / run_id),
)

spec = DatasetSpec(name=cfg["data"]["dataset"], data_dir=str(ROOT / cfg["data"]["data_dir"]), train=True, download=True)
dset = load_dataset(spec)

batch_size = int(cfg["data"]["batch_size"])
num_workers = int(cfg["data"]["num_workers"])

# Split train -> train/val for robust early stopping + generalization monitoring.
val_fraction = float(cfg.get("pretrain", {}).get("val_fraction", 0.1))
val_n = int(round(len(dset) * val_fraction))
train_n = len(dset) - val_n
train_dset, val_dset = torch.utils.data.random_split(
    dset,
    [train_n, val_n],
    generator=torch.Generator().manual_seed(int(cfg["seed"])),
)

loader = DataLoader(train_dset, batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=True)
val_loader = DataLoader(val_dset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
print("train/val sizes:", len(train_dset), len(val_dset))

[data] loading cifar10 (train=True, download=True) from /home/owais/machine unlearning/ebm_unlearning/data
Files already downloaded and verified
train/val sizes: 45000 5000


In [4]:
model = EnergyModel(
    in_channels=int(cfg["model"]["in_channels"]),
    hidden_dim=int(cfg["model"]["hidden_dim"]),
    num_classes=int(cfg["model"].get("num_classes", 10)),
    embed_dim=int(cfg["model"].get("embed_dim", 128)),
    backbone=str(cfg["model"].get("backbone", "conv")),
    finetune_stages=int(cfg["model"].get("finetune_stages", 1)),
    imagenet_pretrained=bool(cfg["model"].get("imagenet_pretrained", True)),
)

from ebm_unlearning.src.training.pretrain import EarlyStoppingConfig

pre_cfg = PretrainConfig(
    epochs=int(cfg["pretrain"]["epochs"]),
    lr=float(cfg["pretrain"]["lr"]),
    weight_decay=float(cfg["pretrain"]["weight_decay"]),
    margin=float(cfg["pretrain"].get("margin", 1.0)),
    k_neg=int(cfg["pretrain"].get("k_neg", 5)),
    neg_chunk=int(cfg["pretrain"].get("neg_chunk", 10)),
    log_every=int(cfg["pretrain"]["log_every"]),
    checkpoint_path=str(ROOT / cfg["pretrain"]["checkpoint_path"]),
)

es_cfg = EarlyStoppingConfig(
    enabled=bool(cfg.get("pretrain", {}).get("early_stopping", {}).get("enabled", False)),
    patience=int(cfg.get("pretrain", {}).get("early_stopping", {}).get("patience", 5)),
    min_delta=float(cfg.get("pretrain", {}).get("early_stopping", {}).get("min_delta", 1e-3)),
    mode=str(cfg.get("pretrain", {}).get("early_stopping", {}).get("mode", "max")),
)


In [5]:
model = pretrain_ebm(
    model,
    loader,
    device=device,
    cfg=pre_cfg,
    logger=logger,
    seed=int(cfg["seed"]),
    tracker=tracker,
    val_loader=val_loader,
    early_stopping=es_cfg,
)
tracker.close()

pretrain epoch 1/50:  85%|████████▌ | 299/351 [00:22<00:03, 15.87it/s][2026-02-03 13:15:13,388] [INFO] [pretrain] step=300 loss=0.627267
[2026-02-03 13:15:18,588] [INFO] [pretrain] epoch_end=1 val_overall_acc=0.668000
[2026-02-03 13:15:18,702] [INFO] [pretrain] saved best checkpoint to /home/owais/machine unlearning/ebm_unlearning/outputs/checkpoints/ebm_pretrained.pt
pretrain epoch 2/50:  99%|█████████▉| 349/351 [00:24<00:00, 13.70it/s][2026-02-03 13:15:43,410] [INFO] [pretrain] step=700 loss=0.657505
[2026-02-03 13:15:45,317] [INFO] [pretrain] epoch_end=2 val_overall_acc=0.677600
[2026-02-03 13:15:45,475] [INFO] [pretrain] saved best checkpoint to /home/owais/machine unlearning/ebm_unlearning/outputs/checkpoints/ebm_pretrained.pt
pretrain epoch 3/50:  85%|████████▍ | 297/351 [00:21<00:03, 13.87it/s][2026-02-03 13:16:06,754] [INFO] [pretrain] step=1000 loss=0.573045
[2026-02-03 13:16:12,194] [INFO] [pretrain] epoch_end=3 val_overall_acc=0.675600
pretrain epoch 4/50:  99%|█████████▉| 3